In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# In this version of the code, I will vectorise over nodes, inputs, and batch size. Also, I will refine the pruning mechanism to include the entropy measure employed in the original paper

In [2]:
class PhyKAN(nn.Module):
    def __init__(self, KANshape, filters_per_unit):
        super().__init__()
        # Find number of layers
        self.Nlayers = len(KANshape)-1
        self.KANshape = KANshape
        # Create parameters for each layer
        self.filter_params = []
        for i in range(self.Nlayers):
            layer_params = 4*(torch.rand((KANshape[i], KANshape[i+1], filters_per_unit, 3))-0.5)
            layer_params.requires_grad=True
            self.filter_params.append(layer_params)
        self.optim = torch.optim.Adam(params=self.filter_params, lr=1e-3)
        self.lossfn = nn.MSELoss()
        
    # Redefined for torch & assuming trainable parameters
    def band_pass(self, Xin, params):
        # Bound parameters
        params=params.sigmoid()
        # Normalise input of 0-1 to frequencies of 100 Hz-100 kHz
        freq = 10**(2+3*Xin).unsqueeze(-1).unsqueeze(-1)
        # For 'sensible' initialisation, we'd like parameters that make sense for gain, f_low, and f_high:
        # Set gain between +/- 5X
        gain = 10*(params[:,:,:, 0].unsqueeze(0)-0.5)
        # Set cutoff frequencies between 10 Hz and 1 MHz
        fc_low = 10+(10**(6*params[:,:,:, 1].unsqueeze(0)))
        fc_high = 10+(10**(6*params[:,:,:, 2].unsqueeze(0)))
        RC_low = (2*torch.pi*fc_low)**-1
        RC_high = (2*torch.pi*fc_high)**-1
        Hout = gain*torch.abs((2j*torch.pi*freq*RC_high/(1+2j*torch.pi*freq*RC_high))*(1/(1+2j*torch.pi*freq*RC_low)))
        return Hout.sum(dim=1).sum(dim=-1)
    
    def forward(self, Xin):
        batch_size = len(Xin)
        # Initialise inputs to and outputs from each layer
        inputs = []
        outputs = []
        for i in range(self.Nlayers):
            inputs.append(torch.zeros((batch_size, self.KANshape[i])))
            outputs.append(torch.zeros((batch_size, self.KANshape[i+1])))
        inputs[0][:, :] = Xin
        # Pass through model
        for layer in range(self.Nlayers):
            outputs[layer][:, :] = self.band_pass(inputs[layer], self.filter_params[layer])      
            if layer < self.Nlayers-1:
                inputs[layer+1][:, :] = outputs[layer][:, :].sigmoid()
        return outputs[-1]
    
    def train(self, Xin, Yin):
        self.optim.zero_grad()
        pred = self.forward(Xin)
        loss = self.lossfn(pred, Yin)
        loss.backward()
        self.optim.step()
        return(loss.item())

    def l1_node(self, l, i, j, n=100):
        xin = torch.linspace(0,1,n)
        l1 = (1/n)*torch.sum(torch.abs(self.band_pass(xin, self.filter_params[l][i,j])))
        return l1

    def l1_layer(self, l):
        layer = self.filter_params[l]
        nin = layer.size[0]
        nout = layer.size[1]
        l1_layer = 0
        for i in range(nin):
            for j in range(nout):
                l1_layer = l1_layer + self.l1_node(l, i, j)
        return l1_layer

    def entropy_layer(self, l):
        layer = self.filter_params[l]
        nin = layer.size[0]
        nout = layer.size[1]
        entropy_layer = 0
        for i in range(nin):
            for j in range(nout):
                entropy_layer = entropy_layer + self.l1_node(l, i, j)/self.l1_layer(l) * torch.log(self.l1_node(l, i, j)/self.l1_layer(l))
        entropy_layer = entropy_layer * -1
        return entropy_layer